# Gordon-fit tinker notebook

Pick a wavelength and a fit recipe. The notebook performs the chosen fit at that single wavelength on the Loisel et al. (2023) Hydrolight dataset and renders:

1. **rrs vs u** with the fit curve overlaid (and standard Gordon as a baseline reference);
2. **(HL − Gordon)/HL residual vs bbp(700 nm)** — bbp(700 nm) is a trophic-state proxy that does not depend on the panel wavelength, so residual structure here is genuinely a function of trophic state.

Supported `fit_type` values:

| value | model | notes |
|---|---|---|
| `'quad'`    | rrs = G1·u + G2·u²                              | 2-parameter baseline |
| `'const'`   | rrs = G0 + G1·u + G2·u²                         | 3-parameter, constant offset |
| `'bbp'`     | rrs = G1·u + G2·u² + Gb·bbp(λ)               | 3-parameter, per-λ bbp slope |
| `'bbp700'`  | rrs = G1·u + G2·u² + Gb·bbp(700)              | 3-parameter, trophic proxy |
| `'joint'`   | rrs = G0 + G1·u + G2·u² + Gb·bbp(λ)          | 4-parameter, joint fit |
| `'2stage'`  | Stage1 (G1,G2); Stage2 (G0,Gb) on residuals vs bbp(700) | 4-parameter, two-stage (this is what the package currently saves to `gordon_coefficients_with_G0_Gb.csv`) |

Edit the **config cell** (cell 4) to change wavelength / fit recipe, then re-run from there.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

from ocpy.hydrolight import loisel23

import calc_gordon as cg
from plot_gordon import plot_rrs_vs_u_single, plot_residual_vs_bbp_single

## Load Loisel23

One-shot load of the elastic dataset. After this cell you have `wave`, `Rrs`, `a`, `bb`, `bbnw`, and `bbp700` available.

In [2]:
ds = loisel23.load_ds(1, 0)
wave_all = ds.Lambda.data
gd = (wave_all >= 350.) & (wave_all <= 750.)
wave = wave_all[gd]
Rrs  = ds.Rrs.data[:, gd]
a    = ds.a.data[:, gd]
bb   = ds.bb.data[:, gd]
bbnw = ds.bbnw.data[:, gd]

# bbp(700) -- single scalar per scene; the trophic-state proxy.
j_700 = int(np.argmin(np.abs(wave - 700.)))
bbp700 = bbnw[:, j_700]

# Pre-compute u and rrs (subsurface) on the full grid.
u_all   = bb / (a + bb)
rrs_all = Rrs / (cg.A_RRS + cg.B_RRS * Rrs)

print(f'Loisel23: {Rrs.shape[0]} scenes, {len(wave)} wavelengths ({wave.min():.0f}-{wave.max():.0f} nm)')
print(f'bbp(700) range: [{bbp700.min():.2e}, {bbp700.max():.2e}] m^-1')

Loisel23: 3320 scenes, 81 wavelengths (350-750 nm)
bbp(700) range: [7.98e-05, 1.34e-02] m^-1


## Configuration — edit me

Set the wavelength and the fit recipe.

In [3]:
# ----- USER KNOBS -----
wv       = 500.0      # nm
fit_type = '2stage'   # one of: 'quad', 'const', 'bbp', 'bbp700', 'joint', '2stage'
# ----------------------

j = int(np.argmin(np.abs(wave - wv)))
wv_actual = wave[j]
u   = u_all[:, j]
rrs = rrs_all[:, j]
sigma = np.maximum(np.abs(rrs), 1e-5)   # relative-weighting sigma
print(f'Selected wavelength: {wv_actual:.1f} nm (idx={j}), fit_type={fit_type!r}')

Selected wavelength: 500.0 nm (idx=30), fit_type='2stage'


## Run the chosen fit

Each branch returns `params` (a dict of fitted coefficients) and `rrs_pred` (the model evaluated at the scene `u` and per-scene `bbp` as appropriate). Edit the branches below to tweak bounds or initial guesses.

In [4]:
def _do_fit(fit_type, u, rrs, sigma, bbp_lambda, bbp700):
    """Single-wavelength fit dispatcher. Returns (params: dict, rrs_pred: array)."""
    if fit_type == 'quad':
        p, _ = curve_fit(cg.rrs_model, u, rrs, p0=(0.1, 0.0), sigma=sigma,
                         bounds=([0.05, -2.0], [0.15, 0.5]))
        params = {'G1': p[0], 'G2': p[1]}
        rrs_pred = cg.rrs_model(u, *p)
    elif fit_type == 'const':
        p, _ = curve_fit(cg.rrs_model_const, u, rrs, p0=(0.0, 0.1, 0.0), sigma=sigma,
                         bounds=([-1e-3, 0.05, -5.0], [1e-3, 0.15, 0.5]))
        params = {'G0': p[0], 'G1': p[1], 'G2': p[2]}
        rrs_pred = cg.rrs_model_const(u, *p)
    elif fit_type in ('bbp', 'bbp700'):
        bbp = bbp_lambda if fit_type == 'bbp' else bbp700
        def _m(X, G1, G2, Gb):
            u_, bbp_ = X
            return cg.rrs_model_bbp(u_, bbp_, G1, G2, Gb)
        p, _ = curve_fit(_m, (u, bbp), rrs, p0=(0.1, 0.0, 0.0), sigma=sigma,
                         bounds=([0.05, -2.0, -1.0], [0.15, 0.5, 1.0]))
        params = {'G1': p[0], 'G2': p[1], 'Gb': p[2]}
        rrs_pred = cg.rrs_model_bbp(u, bbp, *p)
    elif fit_type == 'joint':
        # 4-parameter joint fit with bbp at the panel wavelength.
        def _m(X, G0, G1, G2, Gb):
            u_, bbp_ = X
            return cg.rrs_model_full(u_, bbp_, G0, G1, G2, Gb)
        p, _ = curve_fit(_m, (u, bbp_lambda), rrs, p0=(0.0, 0.1, 0.0, 0.0), sigma=sigma,
                         bounds=([-1e-3, 0.05, -2.0, -1.0], [1e-3, 0.15, 0.5, 1.0]))
        params = {'G0': p[0], 'G1': p[1], 'G2': p[2], 'Gb': p[3]}
        rrs_pred = cg.rrs_model_full(u, bbp_lambda, *p)
    elif fit_type == '2stage':
        # Stage 1: fit (G1, G2).
        p1, _ = curve_fit(cg.rrs_model, u, rrs, p0=(0.1, 0.0), sigma=sigma,
                          bounds=([0.05, -2.0], [0.15, 0.5]))
        # Stage 2: fit (G0, Gb) to residuals vs bbp(700).
        res = rrs - cg.rrs_model(u, *p1)
        def _corr(bbp_, G0, Gb):
            return G0 + Gb * bbp_
        p2, _ = curve_fit(_corr, bbp700, res, p0=(0.0, 0.0), sigma=sigma,
                          bounds=([-1e-3, -1.0], [1e-3, 1.0]))
        params = {'G0': p2[0], 'G1': p1[0], 'G2': p1[1], 'Gb': p2[1]}
        rrs_pred = cg.rrs_model(u, *p1) + _corr(bbp700, *p2)
    else:
        raise ValueError(f'Unknown fit_type: {fit_type!r}')
    return params, rrs_pred

params, rrs_pred = _do_fit(fit_type, u, rrs, sigma, bbnw[:, j], bbp700)
for k, v in params.items():
    print(f'  {k} = {v:+.4e}' if abs(v) < 1e-2 else f'  {k} = {v:+.4f}')

  G0 = +1.3008e-04
  G1 = +0.0977
  G2 = +0.0316
  Gb = -0.1089


## rrs vs u

In [5]:
u_grid = np.linspace(u.min(), u.max(), 200)

# Build the fit overlay. For models that depend on bbp, we evaluate the line
# at the median bbp of the dataset so the overlay is a single representative
# curve through the scatter.
if fit_type == 'quad':
    overlay = cg.rrs_model(u_grid, params['G1'], params['G2'])
    overlay_label = f"fit ({fit_type})  G1={params['G1']:.3f}, G2={params['G2']:+.3f}"
elif fit_type == 'const':
    overlay = cg.rrs_model_const(u_grid, params['G0'], params['G1'], params['G2'])
    overlay_label = f"fit ({fit_type})  G0={params['G0']:+.1e}, G1={params['G1']:.3f}, G2={params['G2']:+.3f}"
elif fit_type == 'bbp':
    bbp_med = float(np.median(bbnw[:, j]))
    overlay = cg.rrs_model_bbp(u_grid, bbp_med, params['G1'], params['G2'], params['Gb'])
    overlay_label = f"fit ({fit_type}) @med bbp({wv_actual:.0f})={bbp_med:.2e}  Gb={params['Gb']:+.2e}"
elif fit_type == 'bbp700':
    bbp_med = float(np.median(bbp700))
    overlay = cg.rrs_model_bbp(u_grid, bbp_med, params['G1'], params['G2'], params['Gb'])
    overlay_label = f"fit ({fit_type}) @med bbp(700)={bbp_med:.2e}  Gb={params['Gb']:+.2e}"
elif fit_type == 'joint':
    bbp_med = float(np.median(bbnw[:, j]))
    overlay = cg.rrs_model_full(u_grid, bbp_med, params['G0'], params['G1'], params['G2'], params['Gb'])
    overlay_label = f"fit ({fit_type}) @med bbp({wv_actual:.0f})={bbp_med:.2e}"
elif fit_type == '2stage':
    bbp_med = float(np.median(bbp700))
    overlay = cg.rrs_model_full(u_grid, bbp_med, params['G0'], params['G1'], params['G2'], params['Gb'])
    overlay_label = f"fit ({fit_type}) @med bbp(700)={bbp_med:.2e}"

fits = [
    {'u_grid': u_grid,
     'rrs_pred': cg.rrs_model(u_grid, cg.G1_STANDARD, cg.G2_STANDARD),
     'label': f'standard  G1={cg.G1_STANDARD}, G2={cg.G2_STANDARD}',
     'color': 'C0', 'ls': '--', 'lw': 1.6},
    {'u_grid': u_grid, 'rrs_pred': overlay,
     'label': overlay_label, 'color': 'C3', 'ls': '-', 'lw': 2},
]

fig, ax = plt.subplots(figsize=(8, 5))
plot_rrs_vs_u_single(ax, u, rrs, fits=fits, log=True)
ax.set_title(f'{wv_actual:.0f} nm  -- {fit_type}')
plt.show()

/tmp/ipykernel_1017333/2678093057.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Residual vs bbp(700)

Reconstruct Rrs (above-surface) from `rrs_pred` so we can compute the same `(HL − Gordon)/HL` percent residual used elsewhere. The x-axis is bbp(700) regardless of the panel wavelength.

In [6]:
Rrs_pred = cg.rrs_to_Rrs(rrs_pred)
res_pct  = 100.0 * (Rrs[:, j] - Rrs_pred) / Rrs[:, j]

fig, ax = plt.subplots(figsize=(8, 5))
plot_residual_vs_bbp_single(
    ax, bbp700, res_pct,
    label=f'{fit_type} @ {wv_actual:.0f} nm',
    color='C3', bbp_wave=700., log_x=True,
)
ax.set_title(f'Residual vs bbp(700)  --  {wv_actual:.0f} nm, {fit_type}')
plt.show()

/tmp/ipykernel_1017333/1628772703.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Stats

In [7]:
rrms_pct = 100.0 * np.sqrt(np.mean(((rrs - rrs_pred) / sigma) ** 2))
print(f'rRMS (rrs space) at {wv_actual:.0f} nm: {rrms_pct:.3f}%')
print(f'mean |residual %| in Rrs space:        {np.mean(np.abs(res_pct)):.3f}%')

# Binned residual vs bbp(700) -- 8 equal-population quantile bins
order = np.argsort(bbp700)
xo = bbp700[order]; ro = res_pct[order]
edges = np.quantile(xo, np.linspace(0, 1, 9))
print(f'\nResidual binned by bbp(700) (8 equal-N bins, N={len(order)//8}/bin):')
for i in range(8):
    sel = (xo >= edges[i]) & (xo < edges[i + 1]) if i < 7 else (xo >= edges[i]) & (xo <= edges[i + 1])
    print(f'  bbp700 [{edges[i]:8.2e}, {edges[i + 1]:8.2e}]  mean_rel%={np.mean(ro[sel]):+6.2f}')

rRMS (rrs space) at 500 nm: 2.480%
mean |residual %| in Rrs space:        1.854%

Residual binned by bbp(700) (8 equal-N bins, N=415/bin):
  bbp700 [7.98e-05, 3.46e-04]  mean_rel%= +4.06
  bbp700 [3.46e-04, 4.72e-04]  mean_rel%= +1.81
  bbp700 [4.72e-04, 5.76e-04]  mean_rel%= +0.57
  bbp700 [5.76e-04, 6.95e-04]  mean_rel%= -0.24
  bbp700 [6.95e-04, 8.24e-04]  mean_rel%= -0.91
  bbp700 [8.24e-04, 1.03e-03]  mean_rel%= -1.48
  bbp700 [1.03e-03, 1.61e-03]  mean_rel%= -2.43
  bbp700 [1.61e-03, 1.34e-02]  mean_rel%= -2.08
